### Clasificación supervisada de imágenes Landsat 8-C2 usando librería Geemap de NASA y datos GEE

In [ ]:
# coding=utf-8
# Título: Práctica 4 - Clasificación supervisada de imágenes
# Requerimientos: Python 3x
# Librerías: geemap, earthengine-api, geopandas, matplotlib
# Autor: Mauricio Tabares Mosquera
# Fecha: 2026-04-16

Instalar librerías

In [ ]:
# !pip install geemap
# !pip install earthengine-api # GEE correcta
# !pip install geopandas
# !pip install matplotlib
# !pip install ee # Instalaba una librería incorrecta
print("Librerías instaladas con éxito")

Importar librerías

In [ ]:
# Importar librerías
import geemap, ee, geopandas as gpd, matplotlib.pyplot as plt, matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

# Autenticar usuario e inicializar el proyecto
ee.Authenticate()
ee.Initialize(project='ee-mautamos')
# Imprimir y validar conexión
print(ee.String('Hola desde los servidores de Google Earth Engine!').getInfo())
print("Librerías importadas con éxito")

Variables de usuario

In [ ]:
# Filtrado de imágenes
cobertura_nube = 10
fecha_inicio = '2024-12-01'
fecha_fin = '2024-12-31'
print("Parámetros para filtar imágenes definidos")

# Mapa base interactivo

# Área de interés para recorte (Formateado desde Copernicus browser)
area_interes = {"type":"Polygon","coordinates":[[[-74.243889,11.186853],[-74.243889,11.262096],[-74.107418,11.262096],[-74.107418,11.186853],[-74.243889,11.186853]]]}
# Zoom
zoom_user = 13
# Combinación
combinacion_rgb = ['B5', 'B4', 'B3']
print("Parámetros de mapa base definidos")

# Entrenamiento

# Definir ruta de archivo GeoJSON
muestras_geojson = r"./muestra.geojson"
print(f"Archivo de muestras: {muestras_geojson}")
# Obtener los nombres de las columnas
muestras = gpd.read_file(muestras_geojson)
print(f"Columnas del archivo: {muestras.columns.tolist()}") # Listar las columnas
# Campo de muestras
campo = "codigo" # Campo entero
print(f"Campo de muestras a utilizar: {campo}")
# Nombre de las bandas por utilizar
bandas = ["B1", "B2", "B3", "B4", "B5", "B6", "B7"]
print(f"Bandas a usar como entrenamiento: {len(bandas)}")
# Número de árboles de decisión
numero_arboles = 100 # El estándar es entre 100 a 500 para no saturar el servicio
print(f"Número de árboles a usar: {numero_arboles}")


Agregar y recortar imagen de satélite Landsat-C2 con mapa base

In [ ]:
# Convertir diccionario a objeto geométrico ee
roi = ee.Geometry(area_interes)

# Filtrar la colección usando el área de interés
coleccion = ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA') \
    .filterBounds(roi) \
    .filterDate(fecha_inicio, fecha_fin) \
    .filterMetadata('CLOUD_COVER', 'less_than', cobertura_nube) \
    .sort("CLOUD_COVER")

# Seleccionar la primera imagen y recortarla
imagen_recortada = coleccion.first().clip(roi)

# Parámetros de visualización
vis_param_1 = {'max': 0.3, 'bands': ['B4', 'B3', 'B2']} # Color real
vis_param_2 = {'max': 0.3, 'bands': combinacion_rgb} # Falso color para vegetación

# Instanciar mapa y añadir capa
mapa = geemap.Map()
# Agregar capas base
mapa.add_basemap('Esri.WorldImagery')
# Agregar capa Landsat recortada
mapa.addLayer(imagen_recortada, vis_param_1, "Recorte - Real [432]")
mapa.addLayer(imagen_recortada, vis_param_2, "Recorte - Falso [543]")

# Centrar el mapa directamente en tu área de interés
mapa.centerObject(roi, zoom_user)

# Mostrar mapa
mapa


Entrenamiento del modelo

In [ ]:
# Convertir GeoJSON local a objeto de Earth Engine
muestras = geemap.geojson_to_ee(muestras_geojson)
print("Archivo de muestras cargado")

# Añadir la capa al mapa
mapa.addLayer(muestras, {}, "Muestras")

# Extraer valores digitales de muestra
areas_entrenamiento = imagen_recortada.select(bandas).sampleRegions(
 **{"collection": muestras, "properties": [campo], "scale": 30}
)
print("Valores de muestra recuperados")


Definir algoritmo de clasificación

In [ ]:
# Entrenamiento

# Support Vector Machine (SVM)
# entrenamiento = ee.Classifier.libsvm().train(areas_entrenamiento, campo, bandas)

# Random Forest
entrenamiento = ee.Classifier.smileRandomForest(numero_arboles).train(areas_entrenamiento, campo, bandas)

# Classification and Regression Trees (CART)
# entrenamiento = ee.Classifier.smileCart().train(areas_entrenamiento, campo, bandas)

# Gradient Tree Boost (GTB)
# entrenamiento = ee.Classifier.smileGradientTreeBoost(100).train(areas_entrenamiento, campo, bandas)

# Minimum Distance (Mínima Distancia)
# entrenamiento = ee.Classifier.minimumDistance('euclidean').train(areas_entrenamiento, campo, bandas)

print("Parámetros de algoritmo definidos")

# Clasificación de la imagen
resultado = imagen_recortada.select(bandas).classify(entrenamiento)
print("Imagen clasificada")

Visualizar imagen clasificada

In [ ]:
# Visualización de resultado

# Paleta de color
paleta = {'min': 1, 'max': 4, 'palette': ['Green', 'Red', 'Blue', 'Yellow']}

# Añadir la capa al mapa, utilizando la paleta
mapa.addLayer(resultado, paleta, "classfied")
print("Mapa creado con éxito")

# Mostrar
mapa

Crear plot estático de resultado

In [ ]:
# Asignar valores desde diccionario
colores_lista = paleta['palette']
min_val = paleta['min']
max_val = paleta['max']

# Convertir el resultado de GEE a matriz para la imagen
img_array = geemap.ee_to_numpy(resultado, region=imagen_recortada.geometry(), scale=30)

# Crear el plot
fig, ax = plt.subplots(figsize=(10, 7))
# Crear el mapa de colores exacto
cmap_gee = ListedColormap(colores_lista)
# Mostrar imagen usando vmin y vmax
im = ax.imshow(img_array, cmap=cmap_gee, vmin=min_val, vmax=max_val)

# Crear leyenda
nombres = ['Bosque', 'Urbano', 'Agua', 'Pasto y cultivo']
parches = [mpatches.Patch(color=colores_lista[i], label=nombres[i]) for i in range(len(nombres))]
ax.legend(handles=parches, loc='upper right', title="Coberturas", frameon=True, facecolor='white', framealpha=0.8)
# Título
ax.set_title(f"Clasificación supervisada de coberturas (Santa Marta, {fecha_inicio[:7]})", fontsize=14)
ax.axis('off')
# Mostrar plot
plt.show()
